# Frameworks: el mismo agente, tres veces

El [capítulo](https://iraitzm.github.io/manual-ia-generativa/parts/agentes/orquestacion.html#frameworks) presenta cinco familias de frameworks y avisa de que lo importante no es cuál gana, sino **qué asume cada una y cuánto acoplamiento aceptáis a cambio**. Y deja tres preguntas para hacerle a cualquier candidato antes de comprometerse:

1. ¿Puedo ver y modificar el contexto exacto que se envía?
2. ¿Emite trazas en un formato estándar?
3. ¿Cuánto código habría que reescribir para cambiar de modelo o de framework?

Este cuaderno coge el agente de la secretaría y lo monta tres veces, para contestar esas preguntas con código delante en lugar de con impresiones.

## Un aviso sobre lo que vais a ver

Este cuaderno tiene un final distinto del que yo esperaba cuando lo empecé. La comparación que uno imagina, *"aquí LangGraph, aquí Agno, mirad qué elegante"*, no llegó a hacerse, porque el primer intento de enchufar un modelo local a un framework **falló en silencio**.

Ese fallo resultó ser mejor material que la comparación. Es exactamente la pregunta 3 de la lista de arriba, respondida por las malas.

## Preparación

In [ ]:
!pip install -q langgraph langchain-huggingface duckdb "transformers>=4.51" torch

In [ ]:
import pathlib
import subprocess
import sys

LOCAL = pathlib.Path("../../data/secretaria")
COLAB = pathlib.Path("manual-ia-generativa/data/secretaria")

if LOCAL.exists():
    base = LOCAL
else:
    if not COLAB.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--quiet",
             "https://github.com/IraitzM/manual-ia-generativa.git"],
            check=True,
        )
    base = COLAB

sys.path.insert(0, str(base.resolve()))

from secretaria import preparar

ctx = preparar()
con = ctx.conectar()

## El caso, una vez

Una sola herramienta y una sola consulta, para que la comparación sea entre frameworks y no entre implementaciones.

In [ ]:
import json
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(ctx.modelo)
modelo = AutoModelForCausalLM.from_pretrained(ctx.modelo, dtype=torch.float32)
modelo.eval()

SISTEMA = ("Eres el asistente de la secretaría académica. Para responder debes llamar "
           "a una de las herramientas disponibles. No respondas de memoria.")
CONSULTA = "¿hasta cuándo puedo pedir la beca?"
PATRON = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)


def consultar_plazo(tramite: str) -> str:
    """Fechas de inicio y fin de un trámite administrativo."""
    filas = con.execute("""
        select tramite, fecha_inicio, fecha_fin from dim_plazo
        where tramite ilike '%' || ? || '%' order by fecha_inicio limit 3
    """, [tramite]).fetchall()
    if not filas:
        return f"No existe el trámite '{tramite}'."
    return "; ".join(f"{t}: del {i} al {f}" for t, i, f in filas)


CATALOGO = {"consultar_plazo": consultar_plazo}

ESQUEMAS = [{"type": "function", "function": {
    "name": "consultar_plazo",
    "description": "Fechas de inicio y fin de un trámite administrativo.",
    "parameters": {"type": "object",
                   "properties": {"tramite": {"type": "string",
                                              "description": "beca, matricula, tfg"}},
                   "required": ["tramite"]}}}]

print(consultar_plazo("beca"))

## Versión 1: sin framework

Es el bucle del [primer cuaderno de agentes](bucle-a-mano.ipynb), recortado a lo mínimo. Lo ponemos aquí para tener la referencia delante.

In [ ]:
def agente_a_mano(consulta, max_vueltas=4):
    mensajes = [{"role": "system", "content": SISTEMA},
                {"role": "user", "content": consulta}]

    for _ in range(max_vueltas):
        texto = tok.apply_chat_template(mensajes, tools=ESQUEMAS, tokenize=False,
                                        add_generation_prompt=True, enable_thinking=False)
        entrada = tok(texto, return_tensors="pt")
        with torch.no_grad():
            salida = modelo.generate(**entrada, max_new_tokens=100, do_sample=False,
                                     pad_token_id=tok.eos_token_id)
        bruto = tok.decode(salida[0][entrada.input_ids.shape[1]:],
                           skip_special_tokens=True).strip()

        encontrado = PATRON.search(bruto)
        if not encontrado:
            return bruto

        llamada = json.loads(encontrado.group(1))
        resultado = CATALOGO[llamada["name"]](**llamada.get("arguments", {}))
        mensajes.append({"role": "assistant", "content": "",
                         "tool_calls": [{"type": "function", "function": llamada}]})
        mensajes.append({"role": "tool", "name": llamada["name"], "content": resultado})

    return "(sin respuesta)"


print(agente_a_mano(CONSULTA))

Veinticinco líneas. Funciona, y **vemos exactamente lo que se envía** porque lo montamos nosotros: `mensajes` está ahí, se puede imprimir, recortar o reordenar en cualquier momento.

Eso es un sí rotundo a la primera pregunta del capítulo. Guardadlo, porque es lo primero que se pierde.

## Versión 2: LangGraph por la vía fácil

LangGraph trae un atajo, `create_react_agent`, que monta el bucle entero. Y `langchain-huggingface` promete envolver un modelo local con la misma interfaz que un modelo de API.

Sobre el papel, esto debería ser el agente en cinco líneas.

In [ ]:
from langchain_core.tools import tool
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline


@tool
def consultar_plazo_lc(tramite: str) -> str:
    """Fechas de inicio y fin de un trámite administrativo."""
    return consultar_plazo(tramite)


llm = HuggingFacePipeline.from_model_id(
    model_id=ctx.modelo, task="text-generation",
    pipeline_kwargs={"max_new_tokens": 100, "do_sample": False},
)
chat = ChatHuggingFace(llm=llm)

respuesta = chat.bind_tools([consultar_plazo_lc]).invoke(CONSULTA)

print("¿ha pedido alguna herramienta?", bool(respuesta.tool_calls))
print("tool_calls:", respuesta.tool_calls)
print("\ncontenido devuelto:")
print(repr(respuesta.content[:400]))

Ahí está.

`bind_tools` no ha protestado. `invoke` no ha lanzado ninguna excepción. La lista `tool_calls` está **vacía**, y el `content` trae el texto en crudo, con los marcadores `<|im_start|>` de la plantilla dentro.

Lo que ha pasado es que esta combinación no sabe convertir el formato de llamadas de Qwen en el objeto `tool_calls` que LangChain espera. La interfaz es la misma que la de un modelo de API; el comportamiento no.

Y lo importante no es el detalle técnico, es **la forma del fallo**: no hay error, hay silencio. Un agente montado así funcionaría, contestaría de memoria y nadie se enteraría hasta ver las respuestas inventadas. Es el peor modo de fallo que puede tener un sistema.

Esta es la respuesta a la tercera pregunta del capítulo, y no la da ningún tutorial: **el acoplamiento a un framework no está en su API, está en qué modelos ha probado de verdad**. Mientras uséis un proveedor de los grandes, todo encaja. En cuanto salís de ahí, aparece el trabajo que la abstracción prometía ahorraros.

## Versión 3: LangGraph explícito

La segunda vía es no usar el atajo: declarar el grafo a mano y meter **nuestra** función de modelo dentro de un nodo. Con esto el framework deja de intentar hablar por nosotros con el modelo y se dedica a lo que sí sabe hacer.

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class Estado(TypedDict):
    mensajes: list
    respuesta: str


def nodo_modelo(estado):
    texto = tok.apply_chat_template(estado["mensajes"], tools=ESQUEMAS, tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)
    entrada = tok(texto, return_tensors="pt")
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=100, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
    bruto = tok.decode(salida[0][entrada.input_ids.shape[1]:],
                       skip_special_tokens=True).strip()

    encontrado = PATRON.search(bruto)
    if not encontrado:
        return {"respuesta": bruto}

    llamada = json.loads(encontrado.group(1))
    return {"mensajes": estado["mensajes"] + [
        {"role": "assistant", "content": "",
         "tool_calls": [{"type": "function", "function": llamada}]}]}


def nodo_herramienta(estado):
    peticion = estado["mensajes"][-1]["tool_calls"][0]["function"]
    resultado = CATALOGO[peticion["name"]](**peticion.get("arguments", {}))
    return {"mensajes": estado["mensajes"] + [
        {"role": "tool", "name": peticion["name"], "content": resultado}]}


def hay_respuesta(estado):
    return END if estado.get("respuesta") else "herramienta"


grafo = StateGraph(Estado)
grafo.add_node("modelo", nodo_modelo)
grafo.add_node("herramienta", nodo_herramienta)
grafo.add_edge(START, "modelo")
grafo.add_conditional_edges("modelo", hay_respuesta,
                            {"herramienta": "herramienta", END: END})
grafo.add_edge("herramienta", "modelo")

agente_grafo = grafo.compile(checkpointer=InMemorySaver())

configuracion = {"configurable": {"thread_id": "alumno-A2023001"}}
resultado = agente_grafo.invoke(
    {"mensajes": [{"role": "system", "content": SISTEMA},
                  {"role": "user", "content": CONSULTA}], "respuesta": ""},
    configuracion,
)

print(resultado["respuesta"])

Funciona, y ahora sí se ve qué se ha comprado con las treinta líneas extra de declaración del grafo.

In [ ]:
estado = agente_grafo.get_state(configuracion)

print("El estado ha sobrevivido a la ejecución:")
print(f"  mensajes guardados: {len(estado.values['mensajes'])}")
print(f"  respuesta:          {estado.values['respuesta'][:60]}")

print("\nY se puede recorrer el historial paso a paso:")
for instantanea in list(agente_grafo.get_state_history(configuracion))[:4]:
    siguiente = instantanea.next or ("(fin)",)
    print(f"  siguiente nodo: {siguiente[0]:12s} mensajes: {len(instantanea.values.get('mensajes', []))}")

Eso es lo que un framework de grafo aporta de verdad, y no es poco:

* **Estado persistente.** El `checkpointer` guarda cada paso. Con uno de disco, un agente puede reanudarse mañana donde lo dejó, o después de un despliegue.
* **Historial recorrible.** Se puede volver a un punto anterior y seguir por otro camino. A mano, eso son varios días de trabajo.
* **El grafo es explícito.** El diagrama y el código son lo mismo, que es justo lo que se pierde cuando el flujo vive esparcido por un `if` dentro de un `for`.

Y fijaos en lo que **no** ha aportado: no ha hablado con el modelo. Esa parte la hemos escrito nosotros, igual que en la versión a mano, y por eso funciona.

Es la lectura más útil del capítulo cuando dice qué conviene no reescribir: *reintentos con criterio, persistencia del estado, trazas y ejecución duradera*. Todo eso está en el grafo. Lo que no conviene delegar a ciegas es la conversación con el modelo, que es donde está vuestro producto.

## Las tres preguntas, contestadas

Con las tres versiones delante ya se puede rellenar la tabla del capítulo con evidencia en lugar de con folletos.

In [ ]:
def contexto_a_mano():
    """Podemos imprimir, byte a byte, lo que se le envía al modelo."""
    mensajes = [{"role": "system", "content": SISTEMA},
                {"role": "user", "content": CONSULTA}]
    return tok.apply_chat_template(mensajes, tools=ESQUEMAS, tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)


enviado = contexto_a_mano()
print(f"1. ¿Veo el contexto exacto?")
print(f"   a mano / grafo explícito: SÍ  ({len(tok(enviado).input_ids)} tokens, imprimible)")
print(f"   create_react_agent:       depende del adaptador, y aquí ni siquiera llegó a enviarlo bien")

print(f"\n2. ¿Trazas estándar?")
print(f"   a mano:            no, hay que instrumentarlo (cuaderno de observabilidad)")
print(f"   grafo:             sí, cada nodo es un punto natural de traza")

print(f"\n3. ¿Cuánto se reescribe al cambiar de modelo?")
print(f"   a mano:            una función, `apply_chat_template` y la expresión regular")
print(f"   grafo explícito:   la misma función; el grafo no se toca")
print(f"   create_react_agent: nada... si el adaptador existe y funciona")

## La tabla del capítulo, revisada

| Familia | Qué compra | Qué cuesta |
|---|---|---|
| **Sin framework** | Control total del contexto. 25 líneas. | Estado, reintentos y trazas, a vuestra costa |
| **De grafo, explícito** | Persistencia, historial recorrible, flujo declarado | 30 líneas de declaración y una dependencia |
| **De grafo, con atajo** | El agente en 5 líneas | Sujeto a que vuestro modelo esté entre los soportados |

La fila que el capítulo defiende y que suele descartarse por prejuicio, la de sin framework, sale bien parada de este ejercicio. No porque los frameworks sean malos, sino porque **el bucle no era la parte difícil**. Lo difícil es el estado, las trazas y la ejecución duradera, y ahí es donde conviene apoyarse en algo maduro.

Una última observación que no estaba en el guion. Los tres montajes usan la misma función para hablar con el modelo, unas quince líneas. Esa función es la que ha decidido si cada versión funcionaba o no. **La pieza que más importa es la que ningún framework os ahorra.**

## Ejercicios

**1. Arreglad el atajo.** Escribid un envoltorio de LangChain que sí parsee el `<tool_call>` de Qwen y pasádselo a `create_react_agent`. Cronometrad cuánto tardáis. Ese tiempo es la respuesta honesta a "¿cuánto cuesta este framework?".

**2. Persistencia de verdad.** Cambiad `InMemorySaver` por el de SQLite, ejecutad el agente, reiniciad el núcleo del cuaderno y comprobad si podéis reanudar la conversación. Es la funcionalidad que más justifica un framework de grafo.

**3. Volver atrás.** Usando `get_state_history`, retroceded al punto anterior a la llamada a la herramienta y reanudad con un resultado distinto, como si la base de datos hubiera contestado otra cosa. Sirve para probar el agente sin tocar los datos.

**4. Un tercer framework.** Coged Agno o Pydantic AI y montad el mismo agente. Antes de escribir código, buscad en su documentación cómo se enchufa un modelo local de `transformers`. Lo que tardéis en encontrarlo ya es un resultado.

**5. Cambiad de modelo.** Sustituid `Qwen3-0.6B` por otro modelo con un formato de llamadas distinto. Contad las líneas que hay que tocar en cada una de las tres versiones. Es la pregunta 3 del capítulo, medida.

**6. El coste de salir.** Coged la versión del grafo e imaginad que hay que quitar LangGraph. ¿Cuántas líneas de lógica de negocio están dentro de sus abstracciones? Si la respuesta es "ninguna", habéis acoplado bien.

## Lo que os lleváis

* **Los fallos de integración son silenciosos.** Ni excepción ni aviso: una lista vacía y una respuesta inventada. Comprobad siempre que vuestro agente **de verdad** llama a las herramientas antes de creeros nada.
* **El acoplamiento de un framework no está en su API**, sino en qué modelos ha probado. Fuera de los proveedores grandes, la abstracción se rompe.
* **El bucle no es la parte difícil.** Son 25 líneas. Elegir framework por el bucle es elegir por lo que menos importa.
* **Lo que sí conviene no reescribir** es el estado persistente, el historial recorrible, los reintentos y las trazas.
* **Escribid vosotros la conversación con el modelo.** Es quince líneas y es lo que decide si funciona.
* Y la pregunta del capítulo, que sigue siendo la buena: **¿podéis ver el contexto exacto que se envía?** Si no, tarde o temprano estaréis depurando a ciegas.

Con esto se cierra la parte de construcción. Lo siguiente es ver por dónde se ha ido cada petición cuando algo falla: [observabilidad](https://iraitzm.github.io/manual-ia-generativa/parts/produccion/observabilidad.html).